# Wikipedia Boilerplate Hard-Negative Benchmark

This notebook demonstrates the **Wikipedia Boilerplate Hard-Negative Benchmark** — a 900-pair dataset built from the Wikipedia API (zero cost, no LLM calls) designed to test near-duplicate text detection methods.

## The Core Challenge

A naive 5-gram Jaccard detector fails because **boilerplate-hard-negative** pairs have high Jaccard similarity (mean ≈ 0.465) due to a shared ~300-400-word CC-BY-SA license header — yet the articles are semantically unrelated. The dataset tests whether a method can distinguish:

| Class | Description | 5-gram Jaccard (mean) |
|---|---|---|
| `near_duplicate` | Original article + splice with 20-30% donor words | 0.582 |
| `boilerplate_hard_negative` | Two unrelated articles with identical license header | 0.465 |
| `random` | Two unrelated articles, no shared header | 0.000 |

The key insight: shared n-grams concentrated in a boilerplate header should not count as evidence of document-level similarity.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# No non-standard packages required — only stdlib + pre-installed packages
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

In [ ]:
import json
import os
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold

## Data Loading

Load the mini benchmark from GitHub (falls back to local file if offline).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-f2202a-edit-clustering-score-spatial-edit-patte/main/round-2/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()

## Configuration

Tunable parameters for the demo experiment. Set to minimum values for a quick run.

In [ ]:
# Number of CV folds (original: 5)
N_FOLDS = 2

# Max examples per class to use (original: 300 per class = all data)
N_PER_CLASS = 5  # demo mini dataset only has 5 per class

# N-gram sizes for Jaccard features
NGRAM_SIZES = [2, 5]  # 2-gram and 5-gram

## Parse Examples

Each example has an `input` field (JSON string with `text_a` and `text_b`), an `output` label, and precomputed metadata (Jaccard scores, boilerplate fraction).

In [ ]:
examples = data["datasets"][0]["examples"]

rows = []
for ex in examples:
    inp = json.loads(ex["input"])
    rows.append({
        "pair_id": ex["metadata_pair_id"],
        "label": ex["output"],
        "fold": ex["metadata_fold"],
        "jaccard_5gram": ex["metadata_jaccard_5gram"],
        "jaccard_2gram": ex["metadata_jaccard_2gram"],
        "boilerplate_frac": ex["metadata_boilerplate_frac"],
        "text_a": inp["text_a"],
        "text_b": inp["text_b"],
    })

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} pairs")
print(df["label"].value_counts())

## Feature Engineering

The key idea: beyond raw Jaccard, compute a **boilerplate-adjusted** Jaccard by identifying n-grams that appear in both texts but are concentrated near the beginning (where boilerplate lives). The adjusted score subtracts shared prefix n-grams from the numerator.

Features:
- `jaccard_5gram`: precomputed 5-gram Jaccard (high for both near-dups AND boilerplate hard-negatives)
- `jaccard_2gram`: precomputed 2-gram Jaccard
- `boilerplate_frac`: fraction of shared n-grams from the prefix region
- `adjusted_jaccard`: raw jaccard × (1 - boilerplate_frac) — the proposed ECS signal

In [ ]:
def get_ngrams(text, n):
    words = text.split()
    return set(zip(*[words[i:] for i in range(n)]))

def jaccard(set_a, set_b):
    if not set_a and not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

def prefix_ngram_frac(text_a, text_b, n, prefix_words=200):
    """Fraction of shared n-grams that come from the first prefix_words words of both texts."""
    words_a = text_a.split()
    words_b = text_b.split()
    prefix_a = set(zip(*[words_a[:prefix_words][i:] for i in range(n)])) if len(words_a) >= n else set()
    prefix_b = set(zip(*[words_b[:prefix_words][i:] for i in range(n)])) if len(words_b) >= n else set()
    all_a = get_ngrams(text_a, n)
    all_b = get_ngrams(text_b, n)
    shared = all_a & all_b
    if not shared:
        return 0.0
    prefix_shared = (prefix_a & prefix_b) & shared
    return len(prefix_shared) / len(shared)

# Compute features
feats = []
for _, row in df.iterrows():
    j5 = row["jaccard_5gram"]
    j2 = row["jaccard_2gram"]
    bp_frac = row["boilerplate_frac"]
    # adjusted: down-weight jaccard by boilerplate concentration
    adj_j5 = j5 * (1.0 - bp_frac)
    feats.append({
        "jaccard_5gram": j5,
        "jaccard_2gram": j2,
        "boilerplate_frac": bp_frac,
        "adjusted_jaccard": adj_j5,
    })

feat_df = pd.DataFrame(feats)
print(feat_df.groupby(df["label"]).mean().round(3))

## Classification Experiment

Compare two classifiers:
1. **Baseline**: raw 5-gram Jaccard only
2. **ECS**: all features including boilerplate-adjusted Jaccard

The hypothesis: the adjusted feature helps separate boilerplate-hard-negatives from true near-duplicates.

In [ ]:
X = feat_df.values
X_baseline = feat_df[["jaccard_5gram"]].values
y = df["label"].values

# Use StratifiedKFold for cross-validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

results = {"baseline": [], "ecs": []}
for train_idx, test_idx in skf.split(X, y):
    for name, Xf in [("baseline", X_baseline), ("ecs", X)]:
        clf = LogisticRegression(max_iter=500, random_state=42)
        clf.fit(Xf[train_idx], y[train_idx])
        acc = clf.score(Xf[test_idx], y[test_idx])
        results[name].append(acc)

for name, accs in results.items():
    print(f"{name:10s}: acc={np.mean(accs):.3f} ± {np.std(accs):.3f}")

## Visualization

Show the distribution of raw vs. adjusted Jaccard by class — the key plot that reveals why boilerplate-adjustment helps.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

labels_order = ["near_duplicate", "boilerplate_hard_negative", "random"]
colors = ["steelblue", "tomato", "seagreen"]

# Plot 1: Raw 5-gram Jaccard by class
ax = axes[0]
for label, color in zip(labels_order, colors):
    vals = df.loc[df["label"] == label, "jaccard_5gram"].values
    ax.scatter(range(len(vals)), vals, label=label, color=color, alpha=0.7)
ax.set_title("Raw 5-gram Jaccard by class")
ax.set_xlabel("Example index")
ax.set_ylabel("Jaccard")
ax.legend(fontsize=7)

# Plot 2: Adjusted Jaccard by class
ax = axes[1]
for label, color in zip(labels_order, colors):
    mask = df["label"] == label
    vals = feat_df.loc[mask, "adjusted_jaccard"].values
    ax.scatter(range(len(vals)), vals, label=label, color=color, alpha=0.7)
ax.set_title("Adjusted Jaccard (×(1−bp_frac)) by class")
ax.set_xlabel("Example index")
ax.set_ylabel("Adjusted Jaccard")
ax.legend(fontsize=7)

# Plot 3: Mean feature values per class (bar chart)
ax = axes[2]
summary = feat_df.groupby(df["label"])["adjusted_jaccard"].mean().reindex(labels_order)
bars = ax.bar(range(len(labels_order)), summary.values, color=colors)
ax.set_xticks(range(len(labels_order)))
ax.set_xticklabels([l.replace("_", "\n") for l in labels_order], fontsize=8)
ax.set_title("Mean adjusted Jaccard per class")
ax.set_ylabel("Mean adjusted Jaccard")
for bar, val in zip(bars, summary.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f"{val:.3f}", ha="center", fontsize=9)

plt.suptitle("Boilerplate Hard-Negative Benchmark: Feature Analysis", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("feature_analysis.png", bbox_inches="tight", dpi=100)
plt.show()

# Summary table
print("\n=== Summary: Mean features by class ===")
summary_all = feat_df.copy()
summary_all["label"] = df["label"].values
print(summary_all.groupby("label").mean().round(3).to_string())

print("\n=== Classification Accuracy ===")
for name, accs in results.items():
    print(f"  {name:10s}: {np.mean(accs):.3f} ± {np.std(accs):.3f}")